In [1]:
import numpy as np
import os
from preprocessing.image_utils import extract_features

def extract_and_save_npy(image_dir, model_cnn, output_dir):
        
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    images_file = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

    features = []
    names = []

    for img_name in (images_file):
        img_path = os.path.join(image_dir, img_name)
        feat = extract_features(model_cnn, img_path)

        features.append(feat)
        names.append(img_name)

    features_matrix = np.array(features)
    names_array = np.array(names)

    np.save(os.path.join(output_dir, "features.npy"), features_matrix)
    np.save(os.path.join(output_dir, "image_names.npy"), names_array)

    print("Selesai! Fitur dan Nama Gambar telah disimpan.")


In [5]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model

image_dir = "dataset/Images"
output_dir = "images_feature"

base_model = InceptionV3(weights='imagenet')
model_cnn = Model(inputs=base_model.input, outputs=base_model.get_layer('avg_pool').output)

features_path = os.path.join(output_dir, "features.npy")
names_path = os.path.join(output_dir, "image_names.npy")

if not os.path.exists(features_path) or not os.path.exists(names_path):
    print("Memulai proses ekstraksi...")
    extract_and_save_npy(image_dir, model_cnn, output_dir)
else:
    print("Load Feature...")


features_matrix = np.load(features_path)
image_names = np.load(names_path)

image_features_map = dict(zip(image_names, features_matrix))


Memulai proses ekstraksi...
Selesai! Fitur dan Nama Gambar telah disimpan.


### Bagian 3

In [ ]:
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, Concatenate, Reshape
from tensorflow.keras.models import Model

image_input = Input(shape=(2048,), name="Image_Input")
caption_input = Input(shape=(35,), name="Caption_Input")

image_projection = Dense(256, activation='relu')(image_input)
image_projection = Reshape((1, 256))(image_projection)

caption_embedding = Embedding(input_dim=vocab_size, output_dim=256)(caption_input)
merged = Concatenate(axis=1)([image_projection, caption_embedding])

x = LSTM(256, return_sequences=True)(merged)
outputs = Dense(vocab_size, activation='softmax')(x)

model = Model(inputs=[image_input, caption_input], outputs=outputs)

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

model.summary()